# Gemma 3 1B (running coach) → LiteRT `.task` for AI Edge Gallery

**Run on Google Colab (GPU runtime).** Does NOT build on macOS-ARM.
Follows Google's official guide: https://ai.google.dev/gemma/docs/conversions/hf-to-mediapipe-task

Package was renamed `ai-edge-torch` → **`litert-torch`**. Gemma 3 needs `kvcache_layout=TRANSPOSED` + `mask_as_input=True`.
Conversion and bundling use **separate installs** (dependency conflict) — in Colab, restart the runtime between Step 1 and Step 2 if imports clash.

In [ ]:
# [1] Merge LoRA adapter into base Gemma 3 1B  (upload adapter.zip first)
!pip -q install transformers peft huggingface_hub safetensors torch
from huggingface_hub import login; login()   # paste HF token (Gemma is gated)
from google.colab import files
import zipfile, glob, torch
up = files.upload()                          # pick adapter.zip
zipfile.ZipFile(list(up)[0]).extractall('adapter_src')
ADAPTER = glob.glob('adapter_src/**/adapter_config.json', recursive=True)[0].rsplit('/',1)[0]
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
BASE='google/gemma-3-1b-it'
m = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16), ADAPTER).merge_and_unload()
m.save_pretrained('merged_gemma3_1b', safe_serialization=True)
AutoTokenizer.from_pretrained(BASE).save_pretrained('merged_gemma3_1b')
print('merged -> merged_gemma3_1b')

In [ ]:
# [2] Convert HF safetensors -> LiteRT .tflite  (int8; use int4 flags if you need smaller)
!pip -q install litert-torch
from litert_torch.generative.examples.gemma3 import gemma3
from litert_torch.generative.utilities import converter
from litert_torch.generative.utilities.export_config import ExportConfig
from litert_torch.generative.layers import kv_cache
model = gemma3.build_model_1b('merged_gemma3_1b')
ec = ExportConfig(); ec.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED; ec.mask_as_input = True
converter.convert_to_tflite(
    model, output_path='.', output_name_prefix='running-coach-gemma3-1b',
    prefill_seq_len=2048, kv_cache_max_len=4096,
    quantize='dynamic_int8', export_config=ec)
import glob; print(glob.glob('*.tflite'))

In [ ]:
# [3] Bundle .tflite + tokenizer -> .task   (restart runtime first if mediapipe import clashes)
!pip -q install mediapipe
from mediapipe.tasks.python.genai import bundler
import glob
tflite = sorted(glob.glob('*.tflite'))[0]
tok = 'merged_gemma3_1b/tokenizer.model'   # Gemma sentencepiece; if missing, download from base repo
cfg = bundler.BundleConfig(
    tflite_model=tflite, tokenizer_model=tok,
    start_token='<bos>', stop_tokens=['<eos>','<end_of_turn>'],
    output_filename='running-coach-gemma3-1b.task',
    prompt_prefix='<start_of_turn>user\n',
    prompt_suffix='<end_of_turn>\n<start_of_turn>model\n')
bundler.create_bundle(cfg)
print('wrote running-coach-gemma3-1b.task')
from google.colab import files; files.download('running-coach-gemma3-1b.task')